# Predicting Educational Video Engagement with Supervised Machine Learning

This notebook builds a supervised machine learning workflow to predict whether educational videos are likely to generate high viewer engagement. The project is based on tabular video-level features such as title length, document entropy, freshness, easiness, stopword presence, normalization rate, speaker speed, and silent period rate (more information in 2. Data Loading)

The aim is to compare baseline and supervised models, account for class imbalance, tune model hyperparameters, and evaluate performance using classification metrics such as ROC AUC, precision, recall, F1 score, ROC curves, precision-recall curves, and confusion matrices.


## 1. Setup

The main libraries used are pandas and NumPy for data handling, matplotlib for plotting, and scikit-learn for preprocessing, model training, model selection, and evaluation.

A fixed random seed is used where applicable to make the workflow more reproducible.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

np.random.seed(0)
RANDOM_STATE = 42


## 2. Data Loading

The notebook expects two preprocessed CSV files:

- `train.csv`, containing features and the target label
- `test.csv`, containing the same features for unseen videos, excluding the target label

The .csv's provided in the `data/` folder are derived from the public `VLE dataset` created by sahanbull, availble here `https://github.com/sahanbull/VLE-Dataset`.

For local use, place these files in your own `data/` folder. To load the files from GitHub instead, replace `DATA_BASE_URL` with the raw GitHub URL for the folder containing `train.csv` and `test.csv`.

Example GitHub raw URL format:

`https://raw.githubusercontent.com/<your-username>/<your-repo>/main/data`


In [ ]:
DATA_BASE_URL = "data"

# Example if hosting the files in your own GitHub repository:
# DATA_BASE_URL = "https://raw.githubusercontent.com/<your-username>/<your-repo>/main/data"

train_data = pd.read_csv(f"{DATA_BASE_URL}/train.csv")
test_data = pd.read_csv(f"{DATA_BASE_URL}/test.csv")

X = train_data.iloc[:, :-1]
y = train_data.iloc[:, -1]

train_data.head()


## 3. Exploratory Data Checks

Before modelling, I check for overlap between the training and test video IDs, inspect the class distribution, and review the feature ranges.

The class distribution is imbalanced, with the negative class making up most of the training examples. The feature summaries also show that variables are on different numeric scales, which matters for models such as logistic regression and SVMs. For those models, scaling should be included inside a pipeline.


In [ ]:
# Check whether any video IDs appear in both train and test data
has_overlap = train_data["id"].isin(test_data["id"]).any()
print(f"Is there overlap between train and test IDs? {has_overlap}")

# Inspect target distribution
print("\nTarget distribution:")
print(y.value_counts())

# Inspect feature ranges and central tendency
traits_df = X.describe().T[["max", "min", "mean", "50%"]]
traits_df


## 4. Train/Validation Split

The training data is split into a training subset and a validation subset. Because the target is imbalanced, I use a stratified split so the class proportions are preserved in both subsets.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)


## 5. Feature Importance Check with a Decision Tree

A decision tree is used as a quick exploratory model to inspect feature importances. Trees do not require feature scaling because they split on feature thresholds directly.

The `id` feature is included in this exploratory step to check whether it appears to encode information. This was done mostly out of curiosity as it is an identifier rather than a meaningful video property, and including identifier columns can sometimes create misleading patterns. At this depth, it does not appear to be an important predictor.


In [ ]:
tree_clf = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
tree_clf.fit(X_train, y_train)

print("Accuracy of Decision Tree classifier on training set: {:.2f}".format(tree_clf.score(X_train, y_train)))
print("Accuracy of Decision Tree classifier on validation set: {:.2f}".format(tree_clf.score(X_val, y_val)))

feat_importances = pd.Series(tree_clf.feature_importances_, index=X_train.columns)
feat_importances = feat_importances.sort_values(ascending=True)

plt.figure(figsize=(10, 6))
feat_importances.plot(kind="barh", color="skyblue", edgecolor="black")
plt.title("Feature Importances (Decision Tree)", fontsize=14, fontweight="bold")
plt.xlabel("Relative Importance Score", fontsize=12)
plt.show()


Id is removed below.

In [ ]:
X_train = X_train.iloc[:, 1:]
X_val = X_val.iloc[:, 1:]

print("Training shape after removing id:", X_train.shape)
print("Validation shape after removing id:", X_val.shape)


## 6. Model Selection with Pipelines

I compare Logistic Regression and SVM models using scikit-learn pipelines. The pipeline ensures that feature scaling is fitted only on the training folds during cross-validation, reducing the risk of data leakage.

ROC AUC is used as the model-selection metric because the classes are imbalanced and I am interested in how well the model ranks likely high-engagement examples above lower-engagement examples.


In [ ]:
# Logistic Regression

start_time = time.time()

pipe_logreg = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

param_grid_logreg = {
    "model__C": [0.1, 1, 10],
}

grid_logreg = GridSearchCV(
    pipe_logreg,
    param_grid=param_grid_logreg,
    cv=5,
    scoring="roc_auc"
)

grid_logreg.fit(X_train, y_train)

end_time = time.time()

print("Took {:.3f} seconds".format(end_time - start_time))
print("Best Logistic Regression C: {:.3f}".format(grid_logreg.best_params_["model__C"]))
print("Best Logistic Regression ROC AUC: {:.3f}".format(grid_logreg.best_score_))


In [ ]:
# Balanced SVM

start_time = time.time()

pipe_svm = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(class_weight="balanced", random_state=RANDOM_STATE))
])

param_grid_svm = {
    "model__C": [0.1, 1, 10],
    "model__gamma": [0.01, 0.1, 1],
    "model__kernel": ["rbf"]
}

grid_svm = GridSearchCV(
    pipe_svm,
    param_grid=param_grid_svm,
    cv=5,
    scoring="roc_auc"
)

grid_svm.fit(X_train, y_train)

end_time = time.time()

print("Took {:.3f} seconds".format(end_time - start_time))
print("Best SVM C: {:.3f}".format(grid_svm.best_params_["model__C"]))
print("Best SVM gamma: {:.3f}".format(grid_svm.best_params_["model__gamma"]))
print("Best SVM ROC AUC: {:.3f}".format(grid_svm.best_score_))


## 7. Validation Set Evaluation

A dummy classifier provides a baseline to show how much value the supervised models add beyond a naive prediction strategy.

The validation results compare accuracy, precision, recall, F1 score, and ROC AUC (For ROC AUC, I use model scores/probabilities rather than hard class predictions. This is important because ROC AUC evaluates ranking quality across thresholds)


In [ ]:
y_pred_logreg = grid_logreg.predict(X_val)
y_pred_svm = grid_svm.predict(X_val)

dummy = DummyClassifier(strategy="stratified", random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_val)

dummy_probs = dummy.predict_proba(X_val)[:, 1]
logreg_scores = grid_logreg.decision_function(X_val)
svm_scores = grid_svm.decision_function(X_val)

results = pd.DataFrame([
    {
        "model": "Dummy",
        "accuracy": accuracy_score(y_val, y_pred_dummy),
        "precision": precision_score(y_val, y_pred_dummy),
        "recall": recall_score(y_val, y_pred_dummy),
        "f1": f1_score(y_val, y_pred_dummy),
        "roc_auc": roc_auc_score(y_val, dummy_probs)
    },
    {
        "model": "Logistic Regression",
        "accuracy": accuracy_score(y_val, y_pred_logreg),
        "precision": precision_score(y_val, y_pred_logreg),
        "recall": recall_score(y_val, y_pred_logreg),
        "f1": f1_score(y_val, y_pred_logreg),
        "roc_auc": roc_auc_score(y_val, logreg_scores)
    },
    {
        "model": "Balanced SVM",
        "accuracy": accuracy_score(y_val, y_pred_svm),
        "precision": precision_score(y_val, y_pred_svm),
        "recall": recall_score(y_val, y_pred_svm),
        "f1": f1_score(y_val, y_pred_svm),
        "roc_auc": roc_auc_score(y_val, svm_scores)
    }
])

results


## 8. Visual Model Evaluation

The confusion matrix shows the classification trade-off between false positives and false negatives. Because the dataset is imbalanced, the ROC curve and precision-recall curve provide additional context beyond accuracy alone.

The balanced SVM is selected as the final model because it provides the strongest validation ROC AUC and substantially improves performance over the dummy baseline.


In [ ]:
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_svm)
plt.title("Confusion Matrix - Balanced SVM")
plt.show()

RocCurveDisplay.from_estimator(grid_svm, X_val, y_val)
plt.title("ROC Curve - Balanced SVM")
plt.show()

PrecisionRecallDisplay.from_estimator(grid_svm, X_val, y_val)
plt.title("Precision-Recall Curve - Balanced SVM")
plt.show()


## 9. Final Model and Unseen Test Predictions

After selecting the final model, I retrain it on the full labelled training dataset. The same feature-preparation decision is used here (the `id` column is excluded from the model inputs and retained only as the output index)

The final output is a probability score for each unseen video, indexed by video ID.


In [ ]:
X_full = train_data.iloc[:, 1:-1]
y_full = train_data.iloc[:, -1]
X_test_final = test_data.iloc[:, 1:]

best_C = grid_svm.best_params_["model__C"]
best_gamma = grid_svm.best_params_["model__gamma"]

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(
        class_weight="balanced",
        C=best_C,
        gamma=best_gamma,
        kernel="rbf",
        probability=True,
        random_state=RANDOM_STATE
    ))
])

start_time = time.time()

final_model.fit(X_full, y_full)
test_probabilities = final_model.predict_proba(X_test_final)[:, 1]

predictions = pd.Series(
    test_probabilities,
    index=test_data["id"],
    name="predicted_engagement_probability"
)

end_time = time.time()

print("Took {:.3f} seconds".format(end_time - start_time))
predictions.head()


## 10. Key Takeaways

- The target distribution is imbalanced, so accuracy alone is not sufficient for model evaluation.
- Pipelines help ensure scaling is applied safely during cross-validation.
- The dummy classifier provides a useful baseline for judging whether the supervised models add value.
- The balanced SVM performed best in validation based on ROC AUC, showing stronger ranking performance than the baseline and Logistic Regression model.
- The final model outputs probability scores for unseen videos, which could be used to rank content by predicted engagement.
